In [1]:
import numpy as np
from scipy.signal import butter, filtfilt, hilbert
from scipy.fft import fft, fftfreq

def compute_bearing_frequencies(rpm, n_balls=9, d=7.938, D=38.5, theta=0):
    """BPFO, BPFI, BSF, FTF in Hz, from shaft RPM + SKF 6205-2RS JEM geometry (CWRU drive-end bearing)."""
    fr = rpm / 60
    ratio = (d / D) * np.cos(np.radians(theta))
    BPFO = (n_balls / 2) * fr * (1 - ratio)
    BPFI = (n_balls / 2) * fr * (1 + ratio)
    BSF  = (D / (2 * d)) * fr * (1 - ratio**2)
    FTF  = (fr / 2) * (1 - ratio)
    return {"BPFO": BPFO, "BPFI": BPFI, "BSF": BSF, "FTF": FTF, "fr": fr}

def envelope_spectrum(signal, fs=48000, band=(2500, 3500)):
    """Bandpass filter around resonance, extract envelope via Hilbert transform, FFT the envelope."""
    nyq = fs / 2
    b, a = butter(4, [band[0]/nyq, band[1]/nyq], btype='band')
    filtered = filtfilt(b, a, signal)

    envelope = np.abs(hilbert(filtered))
    envelope = envelope - envelope.mean()

    n = len(envelope)
    yf = np.abs(fft(envelope))[:n // 2] / n
    xf = fftfreq(n, 1/fs)[:n // 2]
    return xf, yf

def energy_near_frequency(xf, yf, target_freq, window=3):
    """Peak amplitude within +/- window Hz of a target frequency — our actual feature value."""
    mask = (xf > target_freq - window) & (xf < target_freq + window)
    if not mask.any():
        return 0.0
    return yf[mask].max()

In [2]:
signal, rpm = load_mat_file("../data/raw/cwru/B007_1_123.mat")
freqs = compute_bearing_frequencies(rpm)
xf, yf = envelope_spectrum(signal)

for name in ["BPFO", "BPFI", "BSF"]:
    for harmonic in [1, 2, 3]:
        target = freqs[name] * harmonic
        amp = energy_near_frequency(xf, yf, target)
        print(f"{name} x{harmonic} ({target:.1f} Hz): {amp:.6f}")

NameError: name 'load_mat_file' is not defined